# IA para Redes Eletricas Inteligentes / AI for Smart Grids

**Introducao a Inteligencia Artificial — apresentacao dos tres trabalhos praticos**

| # | Trabalho / Assignment | Tecnica / Technique |
|---|---|---|
| 1 | Minissistema especialista | Regras de producao, encadeamento progressivo e regressivo, fatores de certeza |
| 2 | Geracao automatica de planos | STRIPS + GPS (analise meios-fins) + planejamento progressivo por A* |
| 3 | Busca A* | A* com heuristica admissivel, comparada a largura, profundidade, custo uniforme e gulosa |

**Dominio:** a rede de comunicacao que liga ativos distribuidos de um sistema eletrico ao centro de operacao.

> **A topologia e SINTETICA** — modelo didatico da *classe* de cenarios estudada em laboratorios de backhaul sem fio.
> Nao contem inventario real, enderecamento, identificacao de equipamento nem topologia de campo.

**Por que sistemas simbolicos e nao aprendizado de maquina?** Nao existe conjunto de dados rotulado de falhas
para este dominio: rotular um enlace como *degradado* exige instrumento de degradacao controlada e linha de
base de observabilidade autenticada. Sem dados rotulados, o caminho honesto e codificar conhecimento de
engenharia em regras explicitas, **auditaveis e contestaveis**.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from aisg.domain import load_default_topology
from aisg.expert_system import (
    CASES, ConflictResolution, InferenceEngine, build_knowledge_base,
)
from aisg.planning import (
    DIAGNOSIS_TO_FAULT, Operator, Problem, build_restoration_problem,
    format_state, make_state, plan_with_astar, plan_with_gps, problem_from_diagnosis,
)
from aisg.search import RoutingProblem, astar, compare, uniform_cost

LANG = "pt"          # troque para "en" / switch to "en"
topology = load_default_topology()
print(topology.summary(LANG))

---
# 1 — Sistema especialista

## 1.1 A base de conhecimento

43 regras de producao em cinco camadas. As regras de diagnostico nunca leem `rssi_dbm` diretamente,
apenas `signal_quality`: trocar o sensor de RF muda **uma** camada e nada mais.

In [ ]:
kb = build_knowledge_base()
print(f"{len(kb.variables)} variaveis, {len(kb.rules)} regras")
print("problemas de validacao:", kb.validate() or "nenhum")
print("\nlimiares NOMINAIS e NAO CALIBRADOS:")
for name, value in kb.thresholds.items():
    print(f"  {name:<22} {value}")

In [ ]:
# Uma regra de cada tipo / one rule of each kind
for rule_id in ("R01", "R10", "R25", "R36"):
    rule = next(r for r in kb.rules if r.id == rule_id)
    print(rule.render(kb, LANG))
    print(f"     {rule.rationale(LANG)}\n")

## 1.2 Encadeamento progressivo (dirigido por dados)

Parte dos fatos e deriva tudo o que puder — o modo *alarme de monitoramento*.
O ciclo reconhecer-agir dispara **uma** regra por ciclo, para que a ordem do raciocinio fique visivel.

In [ ]:
engine = InferenceEngine(kb)
for variable, value in CASES["interference"].items():
    engine.given(variable, value)

consultation = engine.forward_chain()
print("regras disparadas:", " ".join(consultation.fired_rules), "\n")
for entry in consultation.trace:
    if entry.kind == "fire":
        print(entry.render())

In [ ]:
# Conclusoes com fator de certeza / conclusions with certainty factors
for goal, facts in consultation.conclusions().items():
    print(f"\n{kb.variables[goal].label(LANG)}:")
    for fact in facts:
        bar = "#" * int(round(abs(fact.cf) * 30))
        print(f"  {str(fact.value):<26} {fact.cf:+.2f} {'' if fact.cf >= 0 else '-'}{bar}")

Repare em `rain_fade` com **CF negativo**: tempo limpo e evidencia *contra* atenuacao por chuva (regra R25).
Regras puramente booleanas nao expressariam essa contra-evidencia.

## 1.3 Explicacao: `como` a conclusao foi obtida

In [ ]:
print(consultation.how("diagnosis", "rf_interference", LANG))

## 1.4 Encadeamento regressivo (dirigido por objetivo)

O modo *engenheiro as duas da manha*: pergunta **so** o que o objetivo exige.
Duas propriedades encurtam a consulta: (1) assim que uma condicao e falsa, a regra e abandonada e as
condicoes restantes nao sao perguntadas; (2) quando o objetivo atinge CF 0,9, o motor para.

In [ ]:
case = CASES["power_failure"]
asked = []

def scripted_ask(variable, why_stack):
    asked.append(variable.name)
    value = case.get(variable.name)
    return (value, 1.0) if value is not None else None

backward_engine = InferenceEngine(build_knowledge_base(), ask=scripted_ask)
backward = backward_engine.backward_chain("diagnosis")

askable = [v for v in kb.variables.values() if v.askable]
print(f"perguntas feitas: {len(set(asked))} de {len(askable)} variaveis perguntaveis")
print("  ", ", ".join(dict.fromkeys(asked)))
best = backward.memory.best("diagnosis")
print(f"\ndiagnostico: {best.value} (CF {best.cf:+.2f})")

## 1.5 Resolucao de conflito

Quando varias regras estao prontas, qual dispara primeiro? A politica muda a **ordem do raciocinio**,
nao o ponto fixo.

In [ ]:
for strategy in ConflictResolution:
    e = InferenceEngine(build_knowledge_base(), strategy=strategy)
    for variable, value in CASES["interference"].items():
        e.given(variable, value)
    c = e.forward_chain()
    diagnosis = c.memory.best("diagnosis")
    print(f"{strategy.value:<14} {' '.join(c.fired_rules)}")
    print(f"{'':14} -> {diagnosis.value} (CF {diagnosis.cf:+.2f})\n")

## 1.6 Todos os casos de demonstracao

In [ ]:
print(f"{'caso':<16}{'diagnostico':<26}{'CF':>7}   {'acao recomendada':<24}{'autoriza':>9}")
print("-" * 88)
for name in CASES:
    e = InferenceEngine(build_knowledge_base())
    for variable, value in CASES[name].items():
        e.given(variable, value)
    c = e.forward_chain().conclusions()
    d = c["diagnosis"][0]
    a = c["recommended_action"][0]
    z = c["authorization_required"][0]
    print(f"{name:<16}{str(d.value):<26}{d.cf:>+7.2f}   {str(a.value):<24}{str(z.value):>9}")

---
# 2 — Geracao automatica de planos de acao

## 2.1 STRIPS: a representacao

Cada acao e tres conjuntos: **precondicoes**, **lista de adicao** e **lista de remocao**.
O estado e o conjunto de literais verdadeiros; o que nao esta la e falso (mundo fechado).

In [ ]:
problem = problem_from_diagnosis("node_power_failure", "RM_A5", topology=topology)

op = next(o for o in problem.operators if o.name == "replace_power_unit")
print(f"{op.name}({', '.join(op.parameters)})   custo {op.cost:g}")
print("  precondicoes:", ", ".join(str(p) for p in sorted(op.preconditions, key=str)))
print("  adiciona:    ", ", ".join(str(p) for p in sorted(op.add_list, key=str)))
print("  remove:      ", ", ".join(str(p) for p in sorted(op.delete_list, key=str)))

print("\nESTADO INICIAL"); print(format_state(problem.initial))
print("\nOBJETIVO");       print(format_state(problem.goal))

## 2.2 GPS — analise meios-fins

Olhe a **diferenca** entre o estado atual e o objetivo, escolha um operador que a reduza e,
recursivamente, satisfaca as precondicoes desse operador. Toda acao existe para eliminar
uma diferenca concreta — por isso o trace se le como justificativa, nao como log de busca.

In [ ]:
gps_plan, gps_trace = plan_with_gps(problem, lang=LANG)
print(gps_trace.render()[:1800])

In [ ]:
print(gps_plan.render(LANG))
ok, reason = gps_plan.validate()
print("\nvalidacao:", "plano executavel e atinge o objetivo" if ok else f"FALHOU — {reason}")

## 2.3 Planejador progressivo por A*

Planejamento **e** busca: o estado e o conjunto de literais, o sucessor e qualquer acao aplicavel,
o teste de objetivo e a inclusao do objetivo. Por isso a **mesma funcao `astar`** do trabalho 3
resolve o planejamento — nao uma copia adaptada.

In [ ]:
astar_plan, search_result = plan_with_astar(problem)
print(astar_plan.render(LANG))
print(f"\n{search_result.algorithm}: expandidos {search_result.expanded}, "
      f"gerados {search_result.generated}, pico {search_result.peak_frontier}")

## 2.4 Os dois planejadores em todos os diagnosticos

In [ ]:
print(f"{'diagnostico':<26}{'GPS acoes':>10}{'GPS custo':>11}{'A* acoes':>10}{'A* custo':>10}{'A* exp.':>9}")
print("-" * 76)
for diagnosis in sorted(DIAGNOSIS_TO_FAULT):
    p = problem_from_diagnosis(diagnosis, "RM_A5", topology=topology)
    g, _ = plan_with_gps(p, lang=LANG)
    a, r = plan_with_astar(p)
    print(f"{diagnosis:<26}{g.length:>10}{g.cost:>11g}{a.length:>10}{a.cost:>10g}{r.expanded:>9}")

Repare em `rain_fade`: o plano correto **nao toca a planta e nao desloca equipe**. A causa e
transitoria — intervir seria tratar o clima. E `node_power_failure` custa mais porque exige
deslocar a equipe (custo 4) antes de substituir a fonte (custo 5).

## 2.5 A limitacao do GPS, demonstrada

O GPS ordena os operadores relevantes pelo custo **proprio**, sem olhar o custo das **precondicoes**.

In [ ]:
trap = Problem(
    name="armadilha do GPS",
    operators=[
        Operator.build("conserto_barato_no_local", parameters=("?n",),
                       preconditions=("on-site(?n)",), add=("fixed(?n)",), cost=1.0),
        Operator.build("deslocar_equipe", parameters=("?n",),
                       preconditions=(), add=("on-site(?n)",), cost=10.0),
        Operator.build("conserto_remoto", parameters=("?n",),
                       preconditions=(), add=("fixed(?n)",), cost=3.0),
    ],
    initial=make_state(["broken(N1)"]),
    goal=make_state(["fixed(N1)"]),
    objects={"node": ["N1"]},
    parameter_types={"?n": "node"},
)

g, _ = plan_with_gps(trap, lang=LANG)
a, _ = plan_with_astar(trap)
print(f"GPS: custo {g.cost:g}  ->  {' + '.join(x.name for x in g.actions)}")
print(f"A* : custo {a.cost:g}  ->  {' + '.join(x.name for x in a.actions)}")
print("\nO GPS ve um operador de custo 1 e nao ve o deslocamento de custo 10 que ele exige.")

---
# 3 — Busca A*

## 3.1 A topologia

In [ ]:
import matplotlib.pyplot as plt

KIND_STYLE = {
    "control_centre": ("#1b3a6b", "s", 260),
    "lte_core":       ("#3f7cac", "s", 200),
    "lte_enb":        ("#3f7cac", "^", 200),
    "access_point":   ("#c1666b", "^", 220),
    "remote_master":  ("#6b8f71", "o", 150),
    "saf_relay":      ("#d4a373", "D", 160),
    "substation":     ("#8a6fa8", "p", 200),
    "field_device":   ("#b23a48", "*", 380),
}
LINK_STYLE = {
    "fiber":            ("#1b3a6b", "-",  2.4),
    "ethernet":         ("#4a7c59", "-",  2.0),
    "lte":              ("#3f7cac", "--", 2.0),
    "radio_900mhz":     ("#9a8c78", ":",  1.6),
    "radio_900mhz_saf": ("#d4a373", ":",  2.2),
}

def draw_topology(highlight=None, expanded=None, title=""):
    fig, ax = plt.subplots(figsize=(11, 8))
    for link in topology.active_links():
        a, b = topology.node(link.a), topology.node(link.b)
        colour, style, width = LINK_STYLE[link.type]
        ax.plot([a.x, b.x], [a.y, b.y], style, color=colour, lw=width, alpha=0.65, zorder=1)

    if highlight and len(highlight) > 1:
        xs = [topology.node(n).x for n in highlight]
        ys = [topology.node(n).y for n in highlight]
        ax.plot(xs, ys, "-", color="#e63946", lw=5, alpha=0.85, zorder=2,
                solid_capstyle="round", label="caminho A* / A* path")

    expanded = set(expanded or ())
    for node in topology.nodes.values():
        colour, marker, size = KIND_STYLE[node.kind]
        edge = "#e63946" if highlight and node.id in highlight else "white"
        ax.scatter(node.x, node.y, s=size, c=colour, marker=marker,
                   edgecolors=edge, linewidths=2.0, zorder=3)
        if node.id in expanded:
            ax.scatter(node.x, node.y, s=size * 3.2, facecolors="none",
                       edgecolors="#457b9d", linewidths=1.4, linestyles="--", zorder=2)
        ax.annotate(node.id, (node.x, node.y), textcoords="offset points",
                    xytext=(0, 13), ha="center", fontsize=8.5, zorder=4)

    ax.set_title(title or topology.title(LANG), fontsize=12)
    ax.set_xlabel("metros / metres"); ax.set_ylabel("metros / metres")
    ax.grid(alpha=0.15); ax.set_aspect("equal", adjustable="datalim")
    if highlight and len(highlight) > 1:
        ax.legend(loc="lower left")
    plt.tight_layout()
    return ax

draw_topology(title="Topologia sintetica do backhaul / Synthetic backhaul topology")
plt.show()

## 3.2 Comparacao entre estrategias

O requisito da disciplina: o programa deve retornar **o caminho exato** entre o estado inicial e o objetivo.

In [ ]:
routing = RoutingProblem(topology, "NOC", "RECLOSER_7")
h = routing.heuristic()
results = compare(routing, h)

print(f"{'estrategia':<26}{'passos':>7}{'custo(ms)':>11}{'expand.':>9}{'gerados':>9}{'pico':>7}")
print("-" * 69)
for r in results:
    print(f"{r.algorithm:<26}{r.length:>7}{r.cost:>11.2f}{r.expanded:>9}{r.generated:>9}{r.peak_frontier:>7}")

for r in results:
    print(f"\n{r.algorithm}\n  {' -> '.join(r.path)}")

**Tres licoes numa tabela so:**

1. **A busca em largura minimiza saltos, nao custo.** Acha 4 saltos por ~357 ms — quase 4x o otimo — porque
   conta um salto de fibra e um salto de radio armazena-e-encaminha como iguais.
2. **A gulosa e a mais rapida e esta errada.** Menos da metade das expansoes do A*, caminho ~3,9x mais caro.
   E o argumento mais direto a favor do termo `g(n)`.
3. **O A\* alcanca o mesmo otimo do custo uniforme expandindo menos nos.** A heuristica nao muda a
   resposta; muda o trabalho necessario para chegar a ela.

In [ ]:
best = astar(routing, h)
print(routing.explain_path(best.path, LANG))
draw_topology(highlight=best.path, expanded=best.expansion_order,
              title=f"A*: {' -> '.join(best.path)}  =  {best.cost:.2f} ms")
plt.show()

Circulos tracejados marcam os nos **expandidos** pelo A*. O caminho otimo evita inteiramente a
cadeia armazena-e-encaminha e desce pela sobreposicao LTE.

## 3.3 A heuristica e admissivel e consistente — verificado, nao apenas afirmado

h(n) = distancia em linha reta ate o objetivo, dividida pela maior velocidade efetiva do sistema.

Como toda sobrecarga e nao negativa e nenhuma velocidade supera esse maximo, cada salto custa ao menos
a distancia percorrida dividida pelo maximo. Pela desigualdade triangular, o caminho inteiro custa ao
menos a linha reta dividida pelo maximo. Logo **h nunca superestima**.

In [ ]:
import itertools

worst_ratio, worst_pair = 0.0, None
violations = 0
for source, goal in itertools.permutations(sorted(topology.nodes), 2):
    p = RoutingProblem(topology, source, goal)
    optimal = uniform_cost(p).cost              # verdade de referencia, sem heuristica
    estimate = topology.heuristic(goal)(source)
    if estimate > optimal + 1e-9:
        violations += 1
    if optimal > 0:
        ratio = estimate / optimal
        if ratio > worst_ratio:
            worst_ratio, worst_pair = ratio, (source, goal)

pairs = len(topology.nodes) * (len(topology.nodes) - 1)
print(f"pares testados: {pairs}")
print(f"violacoes de admissibilidade: {violations}")
print(f"razao h/otimo mais alta: {worst_ratio:.3f} em {worst_pair[0]} -> {worst_pair[1]}")
print("\nrazao < 1 em todos os pares  =>  admissivel")

In [ ]:
# Consistencia: h(u) <= custo(u,v) + h(v) para toda aresta e todo objetivo
bad = 0
for goal in topology.nodes:
    hg = topology.heuristic(goal)
    for u in topology.nodes:
        for v, step in topology.successors(u):
            if hg(u) > step + hg(v) + 1e-9:
                bad += 1
print(f"violacoes de consistencia: {bad}   =>  nenhum no precisa ser reaberto")

## 3.4 A vantagem da heuristica CRESCE com o grafo

O cenario de escala tem 30 nos e 44 enlaces: tres setores, cada um com estrela de radio e
cadeia armazena-e-encaminha ate um dispositivo de campo. A resposta continua sendo o caminho
otimo — o que muda e o trabalho para chegar ate ela.

In [ ]:
from aisg.domain import load_topology

scale = load_topology("scale")
print(scale.summary(LANG))

sp = RoutingProblem(scale, "NOC", "FD_C")
print()
print(f"{'estrategia':<26}{'passos':>7}{'custo(ms)':>11}{'expand.':>9}{'gerados':>9}")
print("-" * 62)
for r in compare(sp, sp.heuristic()):
    print(f"{r.algorithm:<26}{r.length:>7}{r.cost:>11.2f}{r.expanded:>9}{r.generated:>9}")

In [ ]:
# Economia agregada sobre TODOS os pares origem-objetivo de cada cenario
import itertools

print(f"{'cenario':<10}{'nos':>5}{'A* exp.':>10}{'UCS exp.':>10}{'economia':>11}")
print("-" * 46)
for name in ("base", "scale"):
    t = load_topology(name)
    a_total = u_total = 0
    for s, g in itertools.combinations(sorted(t.nodes), 2):
        pr = RoutingProblem(t, s, g)
        a_total += astar(pr, pr.heuristic()).expanded
        u_total += uniform_cost(pr).expanded
    print(f"{name:<10}{len(t.nodes):>5}{a_total:>10}{u_total:>10}{1 - a_total / u_total:>10.1%}")

print("\nNum grafo pequeno a busca cega e barata. Num grande, deixa de ser.")

## 3.5 Recalculo de rota sob falha

In [ ]:
topology.disable_link("LTE_ENB", "RM_A5")
degraded = RoutingProblem(topology, "NOC", "RECLOSER_7")
after = astar(degraded, degraded.heuristic())

print(f"antes : {best.cost:>7.2f} ms   {' -> '.join(best.path)}")
print(f"depois: {after.cost:>7.2f} ms   {' -> '.join(after.path)}")
print(f"\ncusto adicional da perda do enlace LTE: {after.cost - best.cost:.2f} ms")

draw_topology(highlight=after.path, title="Rota apos a perda do enlace LTE_ENB-RM_A5")
plt.show()
topology.restore_all_links()

---
# 4 — Integracao: os tres sistemas num incidente

O diagnostico do sistema especialista vira o estado inicial do planejador; o planejador so pode propor
desvio de trafego porque a busca A* **confirmou** que existe rota alternativa evitando o no afetado.

In [ ]:
NODE = "SAF_A2"

# 1 — diagnostico
e = InferenceEngine(build_knowledge_base())
for variable, value in CASES["congestion"].items():
    e.given(variable, value)
conclusions = e.forward_chain().conclusions()
diagnosis = conclusions["diagnosis"][0]
action = conclusions["recommended_action"][0]
print(f"1/3  diagnostico ......... {diagnosis.value} (CF {diagnosis.cf:+.2f})")
print(f"     acao recomendada .... {action.value} (CF {action.cf:+.2f})")

# 2 — planejamento (consulta o A* para saber se existe rota alternativa)
p = problem_from_diagnosis(str(diagnosis.value), NODE, topology=topology)
plan, _ = plan_with_astar(p)
has_alternative = any(str(lit).startswith("alternate-route") for lit in p.initial)
print(f"\n2/3  rota alternativa confirmada pelo A*: {has_alternative}")
print(plan.render(LANG))

# 3 — a rota concreta, evitando o no congestionado
detour = RoutingProblem(topology, "NOC", "RECLOSER_7", avoid=(NODE,))
route = astar(detour, detour.heuristic())
print(f"\n3/3  rota evitando {NODE}: {' -> '.join(route.path)}  =  {route.cost:.2f} ms")

In [ ]:
draw_topology(highlight=route.path, title=f"Trafego desviado de {NODE} / traffic rerouted away from {NODE}")
plt.show()

---
# Limites de validade

Fechar pelos limites, e nao pelos resultados:

- **Os limiares sao nominais e nao calibrados.** Estao reunidos num unico bloco (`THRESHOLDS`) para que a
  calibracao futura ajuste valores **sem reescrever regras**.
- **Nao existe conjunto de dados rotulado de falhas** para este dominio, e nao ha como produzi-lo sem
  instrumento de degradacao controlada e linha de base de observabilidade autenticada.
- **O sistema recomenda, nao atua.** No planejador isso nao e conselho: e precondicao. Agir sem
  autorizacao e um estado **inalcancavel**.
- **A topologia e sintetica.** Uma conclusao obtida aqui vale para **este modelo**, nao para uma planta fisica.

```bash
python -m pytest        # 99 testes
```